# 86 — Downsample `GP*` channels to `DP*` in the position-coded SDS archive

This notebook reads the 1000 Hz `GP?` channels from the position-coded SDS archive, low-pass filters them, downsamples them to 500 Hz, renames them as `DP?`, and writes the converted day files back into the same SDS archive.

Purpose: after this step, the active-source detection notebooks can process all nodes using only `DP*` channels.

Channel mapping:

```text
GPE -> DPE
GPN -> DPN
GPZ -> DPZ
```

Important safeguards:

- The default mode is `DRY_RUN = True`.
- Existing `DP*` day files are not overwritten unless `OVERWRITE = True`.
- AppleDouble files beginning with `._` are ignored.
- A CSV report is written either way.

In [ ]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import numpy as np
from obspy import read
from obspy import Stream

## Configuration

Set `DRY_RUN = False` only after checking the preview report.

In [ ]:
SDS_ROOT = Path("/Volumes/tachyon/LBSSP_DATA/nodal_sds_position_codes")

# Keep True for the first run. Set False to actually write converted day files.
DRY_RUN = False

# Strongly recommended False. If True, converted GP* files can replace existing DP* day files.
OVERWRITE = False

# Anti-alias low-pass before decimation from 1000 Hz to 500 Hz.
LOWPASS_FREQ_HZ = 200.0
LOWPASS_CORNERS = 4

# Expected rates.
INPUT_SAMPLING_RATE_HZ = 1000.0
OUTPUT_SAMPLING_RATE_HZ = 500.0
DECIMATION_FACTOR = 2

CHANNEL_MAP = {
    "GPE": "DPE",
    "GPN": "DPN",
    "GPZ": "DPZ",
}

REPORT_CSV = SDS_ROOT / "downsample_GP_to_DP_report.csv"

print("SDS_ROOT:", SDS_ROOT)
print("DRY_RUN:", DRY_RUN)
print("OVERWRITE:", OVERWRITE)

## Helper functions

In [ ]:
def is_appledouble(path: Path) -> bool:
    """Return True for macOS AppleDouble metadata files."""
    return path.name.startswith("._")


def parse_sds_filename(path: Path) -> dict:
    """
    Parse an SDS day-file name of the form:

        NET.STA.LOC.CHA.D.YEAR.JDAY

    Returns a dictionary with filename fields.
    """
    parts = path.name.split(".")
    if len(parts) < 7:
        raise ValueError(f"Not an SDS day-file name: {path.name}")

    return {
        "network": parts[0],
        "station": parts[1],
        "location": parts[2],
        "channel": parts[3],
        "dtype": parts[4],
        "year": parts[5],
        "jday": parts[6],
    }


def make_sds_output_path(src_path: Path, sds_root: Path, new_channel: str) -> Path:
    """Construct the SDS output path for the converted day file."""
    meta = parse_sds_filename(src_path)

    return (
        sds_root
        / meta["year"]
        / meta["network"]
        / meta["station"]
        / f"{new_channel}.{meta['dtype']}"
        / f"{meta['network']}.{meta['station']}.{meta['location']}.{new_channel}.{meta['dtype']}.{meta['year']}.{meta['jday']}"
    )


def discover_gp_day_files(sds_root: Path) -> list[Path]:
    """Find all GP? SDS day files in the archive."""
    files = []
    for chan in CHANNEL_MAP:
        files.extend(sds_root.rglob(f"*.{chan}.D.*.*"))

    files = [p for p in files if p.is_file() and not is_appledouble(p)]
    return sorted(set(files))


def summarize_sds_station_channels(sds_root: Path, channel_glob: str = "DPZ") -> pd.DataFrame:
    """Summarize available station/location/channel day files in an SDS archive."""
    rows = []
    for p in sorted(sds_root.rglob(f"*.{channel_glob}.D.*.*")):
        if not p.is_file() or is_appledouble(p):
            continue
        try:
            meta = parse_sds_filename(p)
        except Exception:
            continue
        rows.append({**meta, "path": str(p)})
    return pd.DataFrame(rows)

## Preview the files that would be converted

This does not read waveform data yet. It only discovers `GP?` SDS day files and computes the proposed destination path.

In [ ]:
gp_files = discover_gp_day_files(SDS_ROOT)
print(f"Found {len(gp_files)} GP? day files")

preview_rows = []
for src in gp_files:
    meta = parse_sds_filename(src)
    new_channel = CHANNEL_MAP.get(meta["channel"])
    dst = make_sds_output_path(src, SDS_ROOT, new_channel)
    preview_rows.append({
        "network": meta["network"],
        "station": meta["station"],
        "location": meta["location"],
        "old_channel": meta["channel"],
        "new_channel": new_channel,
        "year": meta["year"],
        "jday": meta["jday"],
        "src": str(src),
        "dst": str(dst),
        "dst_exists": dst.exists(),
    })

preview_df = pd.DataFrame(preview_rows)
display(preview_df.head(20))

if len(preview_df):
    display(preview_df.groupby(["network", "location", "old_channel", "new_channel"])["station"].nunique().reset_index(name="n_stations"))
    print("Destination files already exist:", int(preview_df["dst_exists"].sum()))

## Conversion function

For each `GP?` day file:

1. read the MiniSEED file;
2. detrend and taper;
3. low-pass filter at 200 Hz;
4. decimate by 2, producing 500 Hz data;
5. change channel code from `GP?` to `DP?`;
6. write the converted day file to the corresponding `DP?.D` SDS directory.

In [ ]:
def convert_gp_day_file_to_dp(
    src_path: Path,
    sds_root: Path,
    dry_run: bool = True,
    overwrite: bool = False,
    lowpass_freq_hz: float = 200.0,
    lowpass_corners: int = 4,
    expected_input_rate_hz: float = 1000.0,
    output_rate_hz: float = 500.0,
    decimation_factor: int = 2,
) -> dict:
    """Convert one GP? SDS day file to a DP? 500 Hz day file."""
    src_path = Path(src_path)
    sds_root = Path(sds_root)

    out = {
        "src": str(src_path),
        "dst": "",
        "status": "unknown",
        "message": "",
        "network": "",
        "station": "",
        "location": "",
        "old_channel": "",
        "new_channel": "",
        "year": "",
        "jday": "",
        "n_traces": 0,
        "input_sampling_rate_hz": np.nan,
        "output_sampling_rate_hz": np.nan,
    }

    try:
        if is_appledouble(src_path):
            out["status"] = "skip_appledouble"
            return out

        meta = parse_sds_filename(src_path)
        out.update({
            "network": meta["network"],
            "station": meta["station"],
            "location": meta["location"],
            "old_channel": meta["channel"],
            "year": meta["year"],
            "jday": meta["jday"],
        })

        if meta["channel"] not in CHANNEL_MAP:
            out["status"] = "skip_not_gp"
            out["message"] = f"Channel {meta['channel']} is not in CHANNEL_MAP"
            return out

        new_channel = CHANNEL_MAP[meta["channel"]]
        out["new_channel"] = new_channel
        dst_path = make_sds_output_path(src_path, sds_root, new_channel)
        out["dst"] = str(dst_path)

        if dst_path.exists() and not overwrite:
            out["status"] = "exists"
            out["message"] = "Destination exists and overwrite=False"
            return out

        if dry_run:
            out["status"] = "dry_run"
            out["message"] = "Would convert and write"
            return out

        st = read(str(src_path))
        out["n_traces"] = len(st)
        if len(st) == 0:
            out["status"] = "empty_stream"
            return out

        rates = sorted(set(float(tr.stats.sampling_rate) for tr in st))
        out["input_sampling_rate_hz"] = rates[0] if len(rates) == 1 else np.nan

        for tr in st:
            sr = float(tr.stats.sampling_rate)
            if abs(sr - expected_input_rate_hz) > 1e-6:
                raise ValueError(f"Expected {expected_input_rate_hz} Hz but found {sr} Hz in {tr.id}")

            tr.detrend("linear")
            tr.taper(max_percentage=0.02)
            tr.filter("lowpass", freq=lowpass_freq_hz, corners=lowpass_corners, zerophase=True)
            tr.decimate(factor=decimation_factor, no_filter=True)

            if abs(float(tr.stats.sampling_rate) - output_rate_hz) > 1e-6:
                raise ValueError(f"Decimation failed: output rate is {tr.stats.sampling_rate} Hz")

            tr.stats.channel = new_channel

        dst_path.parent.mkdir(parents=True, exist_ok=True)
        st.write(str(dst_path), format="MSEED")

        out["output_sampling_rate_hz"] = float(st[0].stats.sampling_rate)
        out["status"] = "written"
        out["message"] = "Converted successfully"
        return out

    except Exception as e:
        out["status"] = "failed"
        out["message"] = repr(e)
        return out

## Run conversion

First run with `DRY_RUN = True`. If the report looks right, change `DRY_RUN = False` in the configuration cell and rerun from here.

In [ ]:
rows = []

for i, src in enumerate(gp_files, start=1):
    result = convert_gp_day_file_to_dp(
        src,
        sds_root=SDS_ROOT,
        dry_run=DRY_RUN,
        overwrite=OVERWRITE,
        lowpass_freq_hz=LOWPASS_FREQ_HZ,
        lowpass_corners=LOWPASS_CORNERS,
        expected_input_rate_hz=INPUT_SAMPLING_RATE_HZ,
        output_rate_hz=OUTPUT_SAMPLING_RATE_HZ,
        decimation_factor=DECIMATION_FACTOR,
    )
    rows.append(result)

    if i <= 10 or result["status"] in {"failed", "written"}:
        print(f"[{i}/{len(gp_files)}] {result['status']}: {src.name} -> {Path(result['dst']).name if result['dst'] else ''}")
        if result["message"]:
            print("   ", result["message"])

report_df = pd.DataFrame(rows)
REPORT_CSV.parent.mkdir(parents=True, exist_ok=True)
report_df.to_csv(REPORT_CSV, index=False)

print("Report written to:", REPORT_CSV)
if len(report_df):
    display(report_df["status"].value_counts().to_frame("count"))
    display(report_df.head(20))

## Sanity checks

These checks compare `GPZ` availability with `DPZ` availability after conversion. If `DRY_RUN = False`, the `DPZ` station list should include both original 500 Hz D nodes and converted 1000 Hz G nodes.

In [ ]:
gpz_df = summarize_sds_station_channels(SDS_ROOT, channel_glob="GPZ")
dpz_df = summarize_sds_station_channels(SDS_ROOT, channel_glob="DPZ")

print("GPZ day files:", len(gpz_df))
print("DPZ day files:", len(dpz_df))

if len(gpz_df):
    print("
GPZ stations by network/location:")
    display(gpz_df.groupby(["network", "location"])["station"].nunique().reset_index(name="n_stations"))

if len(dpz_df):
    print("
DPZ stations by network/location:")
    display(dpz_df.groupby(["network", "location"])["station"].nunique().reset_index(name="n_stations"))

# Detailed station list for T1 can be useful for the LBSSP line.
for loc in sorted(set(dpz_df["location"]) if len(dpz_df) else []):
    stations = sorted(dpz_df[(dpz_df["network"] == "T1") & (dpz_df["location"] == loc)]["station"].unique())
    if stations:
        print(f"
T1 {loc} DPZ stations: {len(stations)}")
        print(stations)

## Optional: verify one converted file

Set `example_dst` to a file from the report with status `written`, then read it back and check that it is 500 Hz and has a `DP?` channel code.

In [ ]:
written = report_df[report_df["status"].eq("written")].copy() if "report_df" in globals() else pd.DataFrame()

if len(written):
    example_dst = Path(written.iloc[0]["dst"])
    st_test = read(str(example_dst))
    print(example_dst)
    print(st_test)
    print("sampling rates:", sorted(set(tr.stats.sampling_rate for tr in st_test)))
    print("channels:", sorted(set(tr.stats.channel for tr in st_test)))
else:
    print("No files were written in this run. If this was a dry run, set DRY_RUN = False and rerun the conversion cell.")

## Next step

After successful conversion, the refactored `90_*` notebook can process:

```python
channel = "DPZ"       # for detection
channels = "DP?"      # for extraction and SEG-Y shot gathers
```

and it should include both the original 500 Hz D nodes and the downsampled former G nodes.